<div align="center">

# Week 07: Prototypes and Navigation: Menu Trees, Scent and State Machines

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-07) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/Week07_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the three menu trees of the signpost game, the sixteen cards of the card sort and the rules of the paper prototype. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
from collections import deque
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The three menu trees of the signpost game (lab Part A): breadth b and depth d
STRUCTURES = {"4x3": (4, 3), "8x2": (8, 2), "64x1": (64, 1)}
# The sixteen cards of the card sort (lab Part C)
CARDS = ["Book an appointment", "Change an appointment", "Cancel an appointment",
         "Video visit", "Lab results", "Reports", "Vaccinations", "Prescriptions",
         "Directions", "Parking", "Visiting hours", "Pay a bill", "Insurance",
         "Find a doctor", "Ask a nurse", "Feedback"]
# The rules of the paper prototype (lab Part D): (state, tap) -> next state
RULES_PROTO = {
    ("Home", "Book an appointment"): "Departments",
    ("Departments", "Heart, lungs and blood"): "Heart and lungs",
    ("Departments", "Bones, muscles and nerves"): "Bones and muscles",
    ("Departments", "Back"): "Home",
    ("Bones and muscles", "Orthopaedics"): "Days",
    ("Bones and muscles", "Back"): "Departments",
    ("Heart and lungs", "Cardiology"): "Days",
    ("Heart and lungs", "Back"): "Departments",
    ("Days", "Tuesday"): "Times", ("Days", "Wednesday"): "Times",
    ("Days", "Back"): "Heart and lungs",
    ("Times", "10:40"): "Summary", ("Times", "Back"): "Days",
    ("Summary", "Confirm"): "Booked", ("Summary", "Back"): "Times",
    ("Booked", "Home"): "Home"}
print("Setup complete. Running in Colab:", IN_COLAB, "| menu trees:",
      list(STRUCTURES), "| cards:", len(CARDS), "| prototype rules:",
      len(RULES_PROTO))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 7.1": (lambda: 4, 1e-9),
    "Exercise 7.2": (lambda: "Times", 0),
    "Exercise 7.3": (lambda: 2, 1e-9),
    "Exercise A": (lambda: 3 * (1.2 + 0.11 * 4), 1e-6),
    "Exercise B": (lambda: 0.9 ** 3, 1e-6),
    "Exercise C": (lambda: 6 / 8, 1e-6),
    "Exercise D": (lambda: 1 / (1 - 0.25), 1e-6),
    "Exercise E": (lambda: float(STAGE7["novice taps, prototype"]), 1e-6),
}

## Python step 7: Classes and events

An interactive prototype is a machine that waits for events, such as taps, and changes its screen. This step writes such a machine as a *class*, the way Python bundles data with the functions that act on it. It defines a small class first, then a state machine with rules, and feeds it a list of events &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#ref-1)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "RULES_PROTO" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A class with attributes and methods

In [ ]:
# Python practice: a small example for learning the language
class Counter:
    """A counter that remembers its name and how often it was pressed."""
    def __init__(self, name):
        self.name = name
        self.count = 0

    def press(self):
        self.count += 1
        return self.count

book = Counter("Book")
book.press()
book.press()
print(book.name, "was pressed", book.count, "times")

`class Counter:` defines a new kind of object. The method `__init__` runs when an object is created and stores its *attributes*, here `name` and `count`, on `self`, the object itself. The method `press` changes the count. `book = Counter("Book")` creates one object of the class, an *instance*, and `book.press()` calls the method on it.

### A state machine as a class

In [ ]:
# Python practice: a small example for learning the language
class Machine:
    """A current state and rules of the form (state, event) -> next state."""
    def __init__(self, start, rules):
        self.state = start
        self.rules = rules
        self.history = [start]

    def handle(self, event):
        key = (self.state, event)
        if key not in self.rules:
            return False                 # the event does nothing here
        self.state = self.rules[key]
        self.history.append(self.state)
        return True

rules = {("Home", "Book"): "List", ("List", "Cardiology"): "Times",
         ("List", "Back"): "Home", ("Times", "10:40"): "Done"}
m = Machine("Home", rules)
print(m.handle("Book"), m.state)
print(m.handle("Confirm"), m.state)

The rules are a dictionary whose keys are pairs `(state, event)`. The method `handle` looks the pair up. When a rule exists, the machine moves to the next state and records it in `history`. Otherwise the event is ignored and `handle` returns `False`, like a tap on a part of the screen that does nothing.

### Events that change the state

In [ ]:
# Python practice: a small example for learning the language
m = Machine("Home", rules)
events = ["Book", "Back", "Book", "Cardiology", "Help", "10:40"]
ignored = 0
for e in events:
    if not m.handle(e):
        ignored += 1
print("states:", " -> ".join(m.history))
print("final state:", m.state, "| ignored events:", ignored)

A loop feeds the events to the machine one by one, as a user would tap. The history shows the path through the screens, and the counter `ignored` collects the taps that had no effect.

**Quick check.** Why does the machine return `False` instead of stopping with an error when no rule fits?

<details><summary>Answer</summary>

A real interface must survive any tap. An event without a rule simply leaves the screen as it is, and the return value lets the code count such taps.

</details>

### Exercises for Python step 7

**Exercise 7.1.** Create `c = Counter("Next")`, press it four times in a loop and store `c.count` in `presses`.

In [ ]:
presses = None   # your answer

check("Exercise 7.1", presses)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
c = Counter("Next")
for _ in range(4):
    c.press()
presses = c.count

check("Exercise 7.1", presses)

**Exercise 7.2.** Create a new `Machine("Home", rules)`, handle the events "Book" and "Cardiology" and store its final state, a string, in `final`.

In [ ]:
final = None   # your answer

check("Exercise 7.2", final)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m2 = Machine("Home", rules)
for e in ["Book", "Cardiology"]:
    m2.handle(e)
final = m2.state

check("Exercise 7.2", final)

**Exercise 7.3.** Feed the events "Back", "Book", "Help", "Back" to a new machine that starts at "Home", and store the number of ignored events in `n_ignored`.

In [ ]:
n_ignored = None   # your answer

check("Exercise 7.3", n_ignored)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m3 = Machine("Home", rules)
n_ignored = sum(not m3.handle(e) for e in ["Back", "Book", "Help", "Back"])

check("Exercise 7.3", n_ignored)

## 1. Breadth and depth: Your signpost walk &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#ref-5)&#93;

*Lecture: [Menu trees: Breadth and depth](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#breadth-and-depth)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "Machine" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A menu tree with $b$ items per menu and $d$ levels holds $b^d$ items. Python computes the time model of the lecture for every breadth. It then fits the two parameters of the model to the times of your signpost walks with least squares, the method of Week 4.

**From the formulas to the code.** The time of one level $a + cb$ is `a + c * b`. The time of the whole tree $T = d(a + cb)$ is `tree_time(n_items, b, a, c)`, with the depth $d = \ln N / \ln b$ for $N$ items.

In [ ]:
def tree_time(n_items, b, a, c):
    """Time to find one of n_items in a tree of breadth b: d levels of a + c b."""
    d = math.log(n_items) / math.log(b)
    return d * (a + c * b)

for b in [2, 4, 8, 64]:
    print(f"b = {b:>2}: depth {math.log(64) / math.log(b):.0f}, "
          f"T = {tree_time(64, b, 1.2, 0.11):.2f} s")
b_grid = np.linspace(2, 64, 500)
best = b_grid[np.argmin([tree_time(64, b, 1.2, 0.11) for b in b_grid])]
print(f"fastest breadth for a = 1.2 s and c = 0.11 s: {best:.1f}")

**Python note.** `math.log(n_items) / math.log(b)` gives the depth for any breadth, also between whole numbers, so `np.argmin` can search a fine grid of breadths for the fastest tree. Real trees need whole numbers, and the grid only shows where the optimum lies.

**Exercise A.** Store the time of the tree with breadth 4 and depth 3 for $a = 1.2$ s and $c = 0.11$ s in `t_43`.

In [ ]:
t_43 = None   # your answer

check("Exercise A", t_43)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_43 = 3 * (1.2 + 0.11 * 4)

check("Exercise A", t_43)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_SIGNS`. Left empty, the variable makes the code use a simulated class of 24 students who walk each tree four times.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_SIGNS = """
"""

def simulated_walks(n_students=24, a=1.9, c=0.11, seed=RANDOM_STATE):
    """Each student finds four targets in each tree; a level takes a + c b."""
    rng = np.random.default_rng(seed)
    scent = {"4x3": 0.92, "8x2": 0.95, "64x1": 0.99}
    rows = []
    for student in range(1, n_students + 1):
        for name, (b, d) in STRUCTURES.items():
            for target in range(4):
                wrong = sum(rng.random() > scent[name] for _ in range(d))
                levels = d + 2 * wrong          # a wrong turn and the way back
                t = levels * (a + c * b) * rng.lognormal(0, 0.15)
                rows.append([student, name, b, d, target + 1, round(1000 * t), wrong])
    return pd.DataFrame(rows, columns=["student", "structure", "breadth", "depth",
                                       "target", "time_ms", "wrong"])

if MY_SIGNS.strip():
    walks = pd.read_csv(io.StringIO(MY_SIGNS.strip()))
    walks["student"] = 1
    source = "your own walks from the interactive lab"
else:
    walks = simulated_walks()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(walks), "walks")
clean = walks[walks["wrong"] == 0]
per_level = clean["time_ms"] / 1000 / clean["depth"]
c_fit, a_fit = np.polyfit(clean["breadth"], per_level, 1)
print(f"fit: a = {a_fit:.2f} s per level, c = {c_fit:.3f} s per label")
if c_fit <= 0:
    print("c is not positive: The times did not grow with the breadth. More walks give a steadier fit.")
print(walks.groupby("structure")["time_ms"].mean().div(1000).round(2))

**Python note.** Only walks without a wrong turn enter the fit, because a wrong turn adds levels that the model of this section does not count. Dividing each time by the depth gives the time of one level, and `np.polyfit(b, y, 1)` returns the slope $c$ and the intercept $a$ of the best straight line.

In [ ]:
# Figure: measured times against the time model of the tree
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(clean["breadth"], per_level, alpha=0.4, color="#5A4BB5", s=18)
bb = np.geomspace(2, 70, 100)    # many points, so the line bends on the log axis
ax[0].plot(bb, a_fit + c_fit * bb, color="#C98510", label="fitted a + c b")
ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("breadth b (items per menu)")
ax[0].set_ylabel("time per level (s)")
ax[0].legend()
for name, (b, d) in STRUCTURES.items():
    m = clean.loc[clean["structure"] == name, "time_ms"].mean() / 1000
    ax[1].bar(name, m, color="#5A4BB5")
    ax[1].plot(name, d * (a_fit + c_fit * b), "D", color="#C98510")
ax[1].set_ylabel("time to the target (s)")
ax[1].set_title("bars: measured, diamonds: model")
plt.tight_layout()
plt.show()

**Python note.** The left panel uses a logarithmic axis with base 2 for the breadth, so each doubling takes the same width and the straight line $a + cb$ bends upward. `np.geomspace` gives points spaced evenly on this axis. In the right panel the diamonds show the model with the fitted parameters for each tree.

**Observe and reflect.** Which tree was the fastest for you, and does the model agree? Which part of a level does $a$ stand for in the game, and which part would it stand for on a phone?

## 2. Information scent: Wrong turns &#91;[7](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#ref-7)&#93;

*Lecture: [Information scent](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#information-scent)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "c_fit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each level of a tree is a chance to take a wrong turn, and the chances multiply. Python estimates the probability $s$ of a right choice from your walk and computes the share of walks without a wrong turn and the expected time with wrong turns.

**From the formulas to the code.** The probability of the right choice at one level $s$ is `s`, and the share of walks without a wrong turn $s^d$ is `s ** d`. The expected time with wrong turns $\sum_l (a + c b_l)/s_l$ is `time_with_errors(breadths, a, c, s)`.

In [ ]:
def time_with_errors(breadths, a, c, s):
    """Expected time of a path through menus of the given breadths, when each
    level is passed again after a wrong turn with probability 1 - s."""
    s = np.broadcast_to(np.asarray(s, float), (len(breadths),))
    return float(sum((a + c * b) / sl for b, sl in zip(breadths, s)))

print("worked example: s = 0.9, d = 3 ->", round(0.9 ** 3, 3))
print("tree 4x3 with s = 0.9:", round(time_with_errors([4, 4, 4], 1.2, 0.11, 0.9), 2),
      "s")
stats = walks.groupby("structure").agg(walks=("wrong", "size"),
                                       wrong=("wrong", "sum"),
                                       depth=("depth", "first"))
stats["s"] = (stats["walks"] * stats["depth"]) / (
    stats["walks"] * stats["depth"] + stats["wrong"])
stats["no wrong turn"] = stats["s"] ** stats["depth"]
print(stats.round(3))

**Python note.** `np.broadcast_to` turns one value of `s` into one value per level, so the function accepts a single probability or a list. `agg` computes several summaries per tree in one call, each written as `name=(column, function)`.

**Exercise B.** A tree has three levels, and the right label is chosen with $s = 0.9$ at each. Store the probability of a walk without a wrong turn in `p_b`.

In [ ]:
p_b = None   # your answer

check("Exercise B", p_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_b = 0.9 ** 3

check("Exercise B", p_b)

In [ ]:
# Figure: the share of walks without a wrong turn against the depth
d = np.arange(1, 7)
fig, ax = plt.subplots(figsize=(7, 3.4))
for s, col in [(0.99, "#2E8B57"), (0.95, "#5A4BB5"), (0.9, "#C98510"), (0.8, "#B53F2E")]:
    ax.plot(d, s ** d, "o-", color=col, label=f"s = {s}")
for name, row in stats.iterrows():
    ax.plot(row["depth"], row["no wrong turn"], "kx", ms=9)
    ax.annotate(name, (row["depth"], row["no wrong turn"]), xytext=(5, -12),
                textcoords="offset points", fontsize=8)
ax.set_xlabel("depth d (levels)")
ax.set_ylabel("walks without a wrong turn")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** `iterrows` walks through the rows of a table and gives each row with its index, here the name of the tree. The crosses mark the estimates from the walks.

**Observe and reflect.** In which tree did you take the most wrong turns, and at which level? Rewrite one label of that level so that it gives a stronger scent.

## 3. Card sorting: Groups from many sorts &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#ref-9)&#93;

*Lecture: [Card sorting](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#card-sorting)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "stats" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A card sort asks many people to group the same cards. Python counts how often each pair of cards landed in the same group, turns the shares into distances and joins the cards into a tree with hierarchical clustering. Cutting the tree gives the groups of a menu.

**From the formulas to the code.** The share of sorts that put cards $i$ and $j$ together $S_{ij} = n_{ij}/n$ is `S`, and the distance $D_{ij} = 1 - S_{ij}$ is `D`. The average linkage of the lecture is `method="average"` of `linkage`.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score

GROUPS = [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4]
OTHER = {3: 4, 7: 4, 13: 0, 10: 4}   # a second home for unclear cards

def simulated_sorts(n=20, seed=RANDOM_STATE):
    """Each participant puts a card in its usual group, unclear cards often not."""
    rng = np.random.default_rng(seed)
    sorts = []
    for _ in range(n):
        g = np.array(GROUPS)
        for k, alt in OTHER.items():
            if rng.random() < 0.35:
                g[k] = alt
        flip = rng.random(len(g)) < 0.05
        g[flip] = rng.integers(0, 5, flip.sum())
        sorts.append(g)
    return np.array(sorts)

def similarity(sorts):
    """Share of sorts in which each pair of cards shares a group."""
    sorts = np.asarray(sorts)
    return np.mean([np.equal.outer(s, s) for s in sorts], axis=0)

print("worked example:", 3 / 4, "of four participants -> distance", 1 - 3 / 4)
class_sorts = simulated_sorts()
S = similarity(class_sorts)
D = 1 - S
np.fill_diagonal(D, 0)
Z = linkage(squareform(D), method="average")
clusters = fcluster(Z, t=5, criterion="maxclust")
for k in sorted(set(clusters)):
    print(k, [c for c, g in zip(CARDS, clusters) if g == k])

**Python note.** `np.equal.outer(s, s)` compares every card with every other card of one sort. It gives a matrix of `True` and `False`, and the mean over all sorts turns it into shares. `squareform` packs the symmetric distance matrix into the list of pairs that `linkage` expects.

**Exercise C.** In a card sort with eight participants, six put "Parking" and "Directions" into the same group. Store their similarity in `s_c`.

In [ ]:
s_c = None   # your answer

check("Exercise C", s_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s_c = 6 / 8

check("Exercise C", s_c)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_CARDS`. Left empty, the variable makes the code use the first simulated participant as your sort.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_CARDS = """
"""

if MY_CARDS.strip():
    mine = pd.read_csv(io.StringIO(MY_CARDS.strip())).set_index("card")
    my_sort = mine.loc[CARDS, "pile"].to_numpy()
    source = "your own sort from the interactive lab"
else:
    my_sort = class_sorts[0]
    source = "the first simulated participant"
print("Data:", source)
iu = np.triu_indices(16, 1)                     # the 120 pairs of cards
same_me = (my_sort[:, None] == my_sort[None, :])[iu]
same_class = (clusters[:, None] == clusters[None, :])[iu]
agree = int((same_me == same_class).sum())
print(f"pairs on which your sort agrees with the class: {agree} of 120 ({agree / 120:.2f})")
print("adjusted Rand index, the agreement corrected for chance:",
      round(adjusted_rand_score(clusters, my_sort), 3))
S_all = similarity(np.vstack([class_sorts, my_sort]))
pairs = [(S_all[i, j], CARDS[i], CARDS[j]) for i in range(16) for j in range(i + 1, 16)]
unclear = sorted(pairs, key=lambda p: abs(p[0] - 0.5))[:3]
print("the most disputed pairs:")
for s, a, b in unclear:
    print(f"  {a} + {b}: {s:.2f}")

**Python note.** `np.triu_indices(16, 1)` lists the 120 pairs of cards above the diagonal. A pair agrees when both sorts put its two cards together or both keep them apart, the share that Part C of the interactive lab reports. The adjusted Rand index of scikit-learn corrects this share for chance: 1 means the same groups, and 0 means no more agreement than random groups would reach. `np.vstack` adds your sort as one more row below the twenty simulated sorts. Pairs with a similarity near 0.5 are the most disputed: Half of the participants put them together and half did not.

In [ ]:
# Figure: the dendrogram of the card sort
fig, ax = plt.subplots(figsize=(10, 3.8))
dendrogram(Z, labels=CARDS, leaf_rotation=90, ax=ax, color_threshold=0.75)
ax.axhline(0.75, ls="--", color="#585E6E")
ax.set_ylabel("distance 1 - S (average linkage)")
plt.tight_layout()
plt.show()

**Python note.** `dendrogram` draws the order in which the clusters were joined. The height of each join is the average distance between the two clusters, and the dashed line marks where the tree is cut.

**Observe and reflect.** Which of your groups differ from those of the class? Name each group of the dendrogram with a label that a patient would understand.

## 4. State machines: Checking a prototype &#91;[11](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#ref-11)&#93;

*Lecture: [State machines and interactive prototypes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#state-machines)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "clusters" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An interactive prototype is a state machine, so its structure can be checked before any test with users. Python searches the shortest path to the goal and finds screens without a way home. It also computes the expected number of taps of a novice with the Markov model of the lecture.

**From the formulas to the code.** The rules $(s, e) \to s'$ are the dictionary `RULES_PROTO`, the shortest path is `shortest(rules, start, goal)`, and the expected taps $t = (I - Q)^{-1}\mathbf{1}$ are `expected_taps(P, goal)`.

In [ ]:
def neighbours(rules, state):
    return [nxt for (s, e), nxt in rules.items() if s == state]

def shortest(rules, start, goal):
    """Fewest events from start to goal, by breadth-first search."""
    dist, queue = {start: 0}, deque([start])
    while queue:
        s = queue.popleft()
        for nxt in neighbours(rules, s):
            if nxt not in dist:
                dist[nxt] = dist[s] + 1
                queue.append(nxt)
    return dist.get(goal)

def no_way_home(rules, home):
    """States from which no sequence of events leads back to home."""
    states = {s for s, _ in rules} | set(rules.values())
    return sorted(s for s in states if s != home and shortest(rules, s, home) is None)

def expected_taps(P, goal):
    """Expected taps to reach goal from each state of a Markov model P."""
    states = [s for s in P if s != goal]
    idx = {s: i for i, s in enumerate(states)}
    Q = np.zeros((len(states), len(states)))
    for s in states:
        for nxt, p in P[s].items():
            if nxt != goal:
                Q[idx[s], idx[nxt]] += p
    t = np.linalg.solve(np.eye(len(states)) - Q, np.ones(len(states)))
    return {s: round(float(x), 3) for s, x in zip(states, t)}

EXAMPLE = {"H": {"D": 1.0}, "D": {"T": 0.8, "H": 0.2}, "T": {"B": 0.9, "D": 0.1}}
print("worked example:", expected_taps(EXAMPLE, "B"))
print("prototype: shortest path", shortest(RULES_PROTO, "Home", "Booked"),
      "| no way home:", no_way_home(RULES_PROTO, "Home"))

**Python note.** `deque` is a queue that removes items from the front quickly, so the search visits the states in the order of their distance from the start. `np.linalg.solve(A, b)` solves the linear equations $A t = b$ without computing the inverse matrix.

**Exercise D.** A screen leads to the goal with the probability 0.75 per tap and otherwise stays where it is. The expected taps solve $t = 1 + 0.25\,t$. Store $t$ in `t_d`.

In [ ]:
t_d = None   # your answer

check("Exercise D", t_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_d = 1 / (1 - 0.25)

check("Exercise D", t_d)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_PAPER`. Left empty, the variable makes the code use a simulated computer who picks the right paper nine times in ten.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_PAPER = """
"""

SCRIPT = [("Home", "Book an appointment"), ("Departments", "Bones, muscles and nerves"),
          ("Bones and muscles", "Back"), ("Departments", "Heart, lungs and blood"),
          ("Heart and lungs", "Cardiology"), ("Days", "Tuesday"), ("Times", "Back"),
          ("Days", "Wednesday"), ("Times", "10:40"), ("Summary", "Confirm"),
          ("Booked", "Home")]

def simulated_computer(seed=RANDOM_STATE):
    """A computer who shows the right paper with the probability 0.85 per try."""
    rng = np.random.default_rng(seed)
    papers = sorted(set(RULES_PROTO.values()))
    rows = []
    for k, (state, tap) in enumerate(SCRIPT, start=1):
        right = RULES_PROTO[(state, tap)]
        while True:
            ok = rng.random() < 0.85
            chosen = right if ok else rng.choice([p for p in papers if p != right])
            rows.append([k, state, tap, chosen, int(ok),
                         round(1000 * rng.lognormal(np.log(2.5), 0.3))])
            if ok:
                break
    return pd.DataFrame(rows, columns=["event", "state", "tap", "chosen",
                                       "correct", "time_ms"])

if MY_PAPER.strip():
    paper = pd.read_csv(io.StringIO(MY_PAPER.strip()))
    source = "your own session from the interactive lab"
else:
    paper = simulated_computer()
    source = "a simulated computer"
proto = Machine("Home", RULES_PROTO)
for state, tap in SCRIPT:
    proto.handle(tap)
print("Data:", source)
print("the machine went through:", " -> ".join(proto.history))
first = paper.groupby("event").first()
print(f"right paper at the first try: {first['correct'].mean():.2f}, "
      f"wrong papers: {int((paper['correct'] == 0).sum())}, "
      f"the right paper took {paper.loc[paper['correct'] == 1, 'time_ms'].mean() / 1000:.2f} s on average")
print(paper.loc[paper["correct"] == 0, ["event", "state", "tap", "chosen"]])

**Python note.** The class `Machine` of the Python step replays the script of the session. `groupby("event").first()` keeps the first choice for each event, because a wrong paper is followed by another try.

**Observe and reflect.** Where did you show a wrong paper, and why? Which state of the prototype would you check first with a real participant?

## 5. The clinic app, stage 7

*Lecture: [The clinic app, stage 7: A prototype of the new booking flow](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#the-clinic-app-stage-7)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "expected_taps" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 7 checks the prototype of the new booking flow before Week 9 tests it with patients. Python compares the choice of a department in a flat list and in two levels, and it compares the state machines of the current app and of the prototype.

In [ ]:
A, C = 1.2, 0.11          # seconds per screen and per label, as in Week 3
RULES_CURRENT = {
    ("Home", "Services"): "Services", ("Home", "Messages"): "Messages",
    ("Messages", "Back"): "Home", ("Services", "Appointments"): "Appointments",
    ("Services", "Results"): "Results", ("Services", "Back"): "Home",
    ("Results", "Back"): "Services", ("Appointments", "New booking"): "Departments",
    ("Appointments", "My bookings"): "My bookings",
    ("Appointments", "Back"): "Services", ("My bookings", "Back"): "Appointments",
    ("Departments", "Cardiology"): "Doctors", ("Departments", "Other"): "Wrong doctors",
    ("Departments", "Back"): "Appointments", ("Wrong doctors", "Back"): "Departments",
    ("Doctors", "Dr. Arslan"): "Dates", ("Doctors", "Back"): "Departments",
    ("Dates", "10 March"): "Times", ("Dates", "Sunday"): "Closed day",
    ("Dates", "Back"): "Doctors", ("Closed day", "OK"): "Dates",
    ("Times", "10:40"): "Number", ("Times", "Back"): "Dates",
    ("Number", "Book"): "Booked", ("Number", "Wrong number"): "Wrong number",
    ("Wrong number", "OK"): "Number"}
# Probabilities of a novice: the right choice, a wrong turn or a step back
P_CURRENT = {"Home": {"Services": 0.75, "Messages": 0.25}, "Messages": {"Home": 1},
             "Services": {"Appointments": 0.75, "Results": 0.25},
             "Results": {"Services": 1}, "My bookings": {"Appointments": 1},
             "Appointments": {"Departments": 0.9, "My bookings": 0.1},
             "Departments": {"Doctors": 0.8, "Wrong doctors": 0.2},
             "Wrong doctors": {"Departments": 1},
             "Doctors": {"Dates": 0.95, "Departments": 0.05},
             "Dates": {"Times": 0.85, "Closed day": 0.15}, "Closed day": {"Dates": 1},
             "Times": {"Number": 0.92, "Dates": 0.08},
             "Number": {"Booked": 0.88, "Wrong number": 0.12},
             "Wrong number": {"Number": 1}}
P_PROTO = {"Home": {"Departments": 1}, "Bones and muscles": {"Departments": 1},
           "Departments": {"Heart and lungs": 0.9, "Bones and muscles": 0.1},
           "Heart and lungs": {"Days": 0.98, "Departments": 0.02},
           "Days": {"Times": 0.95, "Heart and lungs": 0.05},
           "Times": {"Summary": 0.95, "Days": 0.05},
           "Summary": {"Booked": 0.97, "Times": 0.03}}

STAGE7 = {
    "flat list of 40, expected (s)": round(time_with_errors([40], A, C, 0.80), 2),
    "two levels of 5 and 8, expected (s)": round(time_with_errors([5, 8], A, C,
                                                                  [0.90, 0.98]), 2),
    "no wrong turn, flat list": 0.80,
    "no wrong turn, two levels": round(0.90 * 0.98, 3),
    "shortest path, current (taps)": shortest(RULES_CURRENT, "Home", "Booked"),
    "shortest path, prototype (taps)": shortest(RULES_PROTO, "Home", "Booked"),
    "novice taps, current": float(expected_taps(P_CURRENT, "Booked")["Home"]),
    "novice taps, prototype": float(expected_taps(P_PROTO, "Booked")["Home"]),
    "states without a way home, current": len(no_way_home(RULES_CURRENT, "Home")),
    "states without a way home, prototype": len(no_way_home(RULES_PROTO, "Home"))}
for key, value in STAGE7.items():
    print(f"{key:<42} {value}")
print("trapped in the current app:", no_way_home(RULES_CURRENT, "Home"))

**Python note.** The stage reuses the functions of Sections 2 and 4 without changes: `time_with_errors` for the menus and `shortest`, `no_way_home` and `expected_taps` for the two machines. Only the rules and the probabilities are new.

In [ ]:
# Figure: the menu of departments and the taps of a booking
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(["flat list of 40", "two levels\n5 and 8"],
          [STAGE7["flat list of 40, expected (s)"],
           STAGE7["two levels of 5 and 8, expected (s)"]], color=["#B53F2E", "#2E8B57"])
ax[0].set_ylabel("expected time to the department (s)")
x = np.arange(2)
ax[1].bar(x - 0.2, [STAGE7["shortest path, current (taps)"],
                    STAGE7["novice taps, current"]], 0.4, color="#B53F2E",
          label="current app")
ax[1].bar(x + 0.2, [STAGE7["shortest path, prototype (taps)"],
                    STAGE7["novice taps, prototype"]], 0.4, color="#2E8B57",
          label="prototype")
ax[1].set_xticks(x, ["shortest path", "novice, expected"])
ax[1].set_ylabel("taps from Home to Booked")
ax[1].legend()
plt.tight_layout()
plt.show()

**Python note.** The two panels repeat the values of `STAGE7`. The left one compares the menus, the right one the state machines.

**Exercise E.** Store the expected taps of a novice in the prototype, as kept in `STAGE7`, in `taps_e`.

In [ ]:
taps_e = None   # your answer

check("Exercise E", taps_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
taps_e = STAGE7["novice taps, prototype"]

check("Exercise E", taps_e)

**Observe and reflect.** Which states trap the patients of the current app, and what would you add to each of them? Which probability of the novice model would you measure first in Week 9?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE7" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-07/"
               "NB07_prototypes_and_navigation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below has a menu tree for one frequent task, as it is now and as a redesign. The function of Section 2 predicts the expected time of both after one change of a name.

The breadths, depths and probabilities are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "e_government"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {"clinic": ("choose a department", [40], [0.80], [5, 8], [0.90, 0.98]),
        "banking": ("find the limit of a card", [6, 5, 7], [0.85, 0.8, 0.9],
                    [8, 6], [0.92, 0.95]),
        "e_government": ("download a tax document", [12, 9, 6, 5],
                         [0.85, 0.8, 0.85, 0.9], [9, 12], [0.9, 0.95]),
        "food_delivery": ("find a vegetarian dish", [20, 30], [0.9, 0.85],
                          [8, 6, 10], [0.95, 0.95, 0.97]),
        "course_registration": ("find an elective course", [5, 7, 6, 9],
                                [0.9, 0.8, 0.85, 0.9], [7, 10], [0.92, 0.95])}

task, b_now, s_now, b_new, s_new = APPS[APP]
t_now = time_with_errors(b_now, 1.2, 0.11, s_now)
t_new = time_with_errors(b_new, 1.2, 0.11, s_new)
print(f"{APP}: {task}")
print(f"  current menus {b_now}: {t_now:.2f} s, without a wrong turn "
      f"{np.prod(s_now):.2f}")
print(f"  redesign {b_new}: {t_new:.2f} s, without a wrong turn {np.prod(s_new):.2f}")

**Python note.** Each entry holds the task, the breadths and probabilities of the current tree and those of the redesign. `np.prod` multiplies the probabilities of all levels, the share of walks without a wrong turn.

**Observe and reflect.** Is the redesign of your app faster because it is broader, because its labels are clearer, or both? Which label of the current tree would you test first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the screens of a command-line tool that you use as a state machine with the class `Machine`, and check it with `no_way_home`. |
| Web development | Draw the navigation tree of a website with more than thirty pages and compute its time with `time_with_errors` for a broader version. |
| Mobile development | Model the settings menu of a phone as a tree, estimate the probability of a right choice for three labels and compare two depths. |
| Game development | Write the pause menu of a game as a state machine and compute the expected presses of a novice who looks for the volume setting. |
| Embedded systems and the Internet of Things | Write the menu of a washing machine as a state machine and find every state from which the start screen cannot be reached. |
| Data science | Run a card sort with five friends on twelve cards of a news site and cluster the results with the code of Section 3. |
| Cybersecurity | Model the steps of recovering a password as a state machine and look for states where a user is trapped or can skip a check. |
| Health informatics | Model the alarm menu of an infusion pump as a state machine and compute the expected presses of a nurse in a hurry. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-07).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>